In [1]:
# Train the TCR-epitope model from the notebook (may take several minutes)
import sys
# ensure current directory is on path
sys.path.insert(0, '.')
from train_model_inductive import train
# Train the hetero GNN to match notebook cell 6 graph construction
train('train.tsv', 'test.tsv', out_model='models/tcr_epitope_model_inductive.pt', model_type='gnn', epochs=100)

/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch 1/100  train_loss=0.6008  test_auc=0.7152  test_acc=0.6421
Epoch 2/100  train_loss=0.5327  test_auc=0.7405  test_acc=0.6632
Epoch 3/100  train_loss=0.5000  test_auc=0.7490  test_acc=0.6667
Epoch 4/100  train_loss=0.4724  test_auc=0.7586  test_acc=0.6790
Epoch 5/100  train_loss=0.4447  test_auc=0.7635  test_acc=0.6790
Epoch 6/100  train_loss=0.4192  test_auc=0.7706  test_acc=0.6865
Epoch 7/100  train_loss=0.3950  test_auc=0.7711  test_acc=0.6838
Epoch 8/100  train_loss=0.3709  test_auc=0.7791  test_acc=0.6949
Epoch 9/100  train_loss=0.3480  test_auc=0.7736  test_acc=0.6928
Epoch 10/100  train_loss=0.3197  test_auc=0.7846  test_acc=0.6990
Epoch 11/100  train_loss=0.2970  test_auc=0.7822  test_acc=0.6980
Epoch 12/100  train_loss=0.2724  test_auc=0.7870  test_acc=0.7056
Epoch 13/100  train_loss=0.2512  test_auc=0.7752  test_acc=0.6952
Epoch 14/100  train_loss=0.2319  test_auc=0.7773  test_acc=0.6970
Epoch 15/100  train_loss=0.2105  test_auc=0.7838  test_acc=0.7065
Epoch 16/100  train

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def make_negatives(
    positives: pd.DataFrame,
    neg_ratio: int = 1,
    seed: int = 0,
) -> pd.DataFrame:
    """
    positives must contain columns: cdr3, peptide
    Returns a combined labeled df with columns: cdr3, peptide, label
      label=1 for positives, label=0 for sampled negatives
    Negatives are sampled so they are not in the positive pair set.
    """
    rng = np.random.default_rng(seed)

    pos = positives[["cdr3", "peptide"]].dropna().drop_duplicates().copy()
    pos["label"] = 1

    pos_pairs = set(map(tuple, pos[["cdr3", "peptide"]].values.tolist()))
    cdr3s = pos["cdr3"].unique()
    peps = pos["peptide"].unique()

    n_pos = len(pos)
    n_neg_target = n_pos * int(neg_ratio)

    neg_rows = []
    attempts = 0
    max_attempts = n_neg_target * 50 + 10_000  # safety

    while len(neg_rows) < n_neg_target and attempts < max_attempts:
        attempts += 1
        c = rng.choice(cdr3s)
        p = rng.choice(peps)
        if (c, p) in pos_pairs:
            continue
        neg_rows.append((c, p))

    neg = pd.DataFrame(neg_rows, columns=["cdr3", "peptide"]).drop_duplicates()
    neg["label"] = 0

    combined = pd.concat([pos, neg], ignore_index=True)
    return combined

def metrics_from_probs(df_with_label: pd.DataFrame, threshold: float = 0.5) -> dict:
    valid = df_with_label[df_with_label["label"].notna() & df_with_label["pred_prob"].notna()].copy()
    y_true = valid["label"].astype(int).values
    y_prob = valid["pred_prob"].values
    y_pred = (y_prob >= threshold).astype(int)

    out = {
        "n": len(valid),
        "pos": int(y_true.sum()),
        "neg": int((1 - y_true).sum()),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    if len(np.unique(y_true)) == 2:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
    else:
        out["roc_auc"] = np.nan
    return out

def evaluate_positive_only_dataset(
    positives_df: pd.DataFrame,
    predict_fn,                 # function that takes df(cdr3,peptide) and returns df with pred_prob
    name: str,
    neg_ratio: int = 1,
    threshold: float = 0.5,
    seed: int = 0,
):
    labeled = make_negatives(positives_df, neg_ratio=neg_ratio, seed=seed)
    labeled_pred = predict_fn(labeled)   # must add pred_prob
    m = metrics_from_probs(labeled_pred, threshold=threshold)
    m["dataset"] = name
    m["neg_ratio"] = neg_ratio
    return m


In [3]:
import numpy as np
import pandas as pd
import torch
from torch import nn

import chephy_model as cpm

# PyG
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

# -------------------------
# Model (same as training)
# -------------------------
class HeteroGNN(nn.Module):
    def __init__(self, tcr_in: int, epi_in: int, hidden: int = 256, num_layers: int = 2):
        super().__init__()
        self.lin_t = nn.Linear(tcr_in, hidden)
        self.lin_e = nn.Linear(epi_in, hidden)

        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(
                HeteroConv(
                    {
                        ('tcr', 'similar', 'tcr'): SAGEConv((-1, -1), hidden),
                        ('tcr', 'binds', 'epitope'): SAGEConv((-1, -1), hidden),
                        ('epitope', 'rev_binds', 'tcr'): SAGEConv((-1, -1), hidden),
                    },
                    aggr="sum",
                )
            )

        self.act = nn.ReLU()
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x_dict, edge_index_dict):
        x_dict = {
            "tcr": self.lin_t(x_dict["tcr"]),
            "epitope": self.lin_e(x_dict["epitope"]),
        }
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {k: self.act(v) for k, v in x_dict.items()}
        return x_dict

# -------------------------
# Helpers
# -------------------------
def prepare_pairs_df(df: pd.DataFrame) -> pd.DataFrame:
    """Same preprocessing as training: AA-only, len>=8, truncate to 8."""
    out = df.copy()
    out = out.dropna(subset=["cdr3", "peptide"])
    out = out[out["cdr3"].str.match(r"^[ARNDCEQGHILKMFPSTWYV]+$", na=False)]
    out = out[out["cdr3"].str.len() >= 8]
    out = cpm.truncate_sequences(out, "cdr3", right=4, left=4)  # adds cdr3_truncated
    out = out.dropna(subset=["cdr3_truncated"])
    return out

def scaler_from_state(state: dict):
    """Rebuild sklearn StandardScaler from saved mean_/scale_."""
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler(with_mean=True, with_std=True)
    sc.mean_ = np.asarray(state["mean_"], dtype=np.float64)
    sc.scale_ = np.asarray(state["scale_"], dtype=np.float64)
    sc.n_features_in_ = sc.mean_.shape[0]
    return sc

def build_inductive_graph_and_embeddings(
    train_path: str,
    predict_dfs: dict,                 # name -> df with columns cdr3, peptide
    model_path: str = "models/tcr_epitope_model_inductive.pt",
    hidden: int = 256,
    num_layers: int = 2,
):
    """
    Builds an inductive hetero-graph:
      - nodes: train TCR/EPI + unseen from all predict_dfs
      - edges:
         * ('tcr','similar','tcr') from kNN over ALL TCR nodes (train+unseen)
         * ('tcr','binds','epitope') ONLY from train pairs (no leakage)
         * reverse binds
    Then runs model forward to produce node embeddings.
    Returns maps + embeddings dict + device.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(model_path, map_location=device)

    if ckpt.get("model_type") not in ("gnn_hetero", "gnn"):
        raise ValueError(f"Checkpoint model_type={ckpt.get('model_type')} is not a hetero GNN.")

    knn_k = int(ckpt.get("knn_k", 10))
    atchley = cpm.load_atchley_map()

    # scalers (train-fitted)
    tcr_scaler = scaler_from_state(ckpt["tcr_scaler_state"])
    epi_scaler = scaler_from_state(ckpt["epi_scaler_state"])

    # target_length for epitopes (pad) = (feature_dim / 5)
    epi_dim = int(epi_scaler.n_features_in_)
    target_length = max(1, epi_dim // 5)

    # Load + preprocess train
    train_df = pd.read_csv(train_path, sep="\t")
    train_df = prepare_pairs_df(train_df)

    # Preprocess predict dfs
    pred_prepped = {name: prepare_pairs_df(df) for name, df in predict_dfs.items()}

    # Build node vocab: start with TRAIN unique
    train_tcrs = list(dict.fromkeys(train_df["cdr3_truncated"].tolist()))
    train_epis = list(dict.fromkeys(train_df["peptide"].tolist()))

    # Collect unseen from ALL prediction datasets (union)
    all_pred_tcrs = []
    all_pred_epis = []
    for dfp in pred_prepped.values():
        all_pred_tcrs.extend(dfp["cdr3_truncated"].tolist())
        all_pred_epis.extend(dfp["peptide"].tolist())

    all_pred_tcrs = list(dict.fromkeys(all_pred_tcrs))
    all_pred_epis = list(dict.fromkeys(all_pred_epis))

    unseen_tcrs = [s for s in all_pred_tcrs if s not in set(train_tcrs)]
    unseen_epis = [s for s in all_pred_epis if s not in set(train_epis)]

    all_tcrs = train_tcrs + unseen_tcrs
    all_epis = train_epis + unseen_epis

    # Build feature matrices INDUCTIVELY: transform using train scalers (no refit)
    tcr_feats_all, _ = cpm.sequences_to_features(
        all_tcrs,
        atchley=atchley,
        standardize=True,
        scaler=tcr_scaler,
    )
    epi_feats_all, _ = cpm.sequences_to_features_any_length(
        all_epis,
        atchley=atchley,
        standardize=True,
        scaler=epi_scaler,
        agg="pad",
        target_length=target_length,
    )

    # Index maps
    tcr_to_idx = {s: i for i, s in enumerate(all_tcrs)}
    epi_to_idx = {s: i for i, s in enumerate(all_epis)}

    # Map train indices (for binds edges)
    train_df = train_df.copy()
    train_df["tcr_index"] = train_df["cdr3_truncated"].map(tcr_to_idx).astype(int)
    train_df["epitope_index"] = train_df["peptide"].map(epi_to_idx).astype(int)

    # Build kNN similarity over ALL TCR nodes (train+unseen)
    # Use the same helper + same k, using scaler for standardize_features distance space.
    tcr_edge_index, tcr_edge_weight, _, _ = cpm.build_patient_knn_graph(
        all_tcrs,
        k=knn_k,
        atchley_map=atchley,
        standardize_features=True,
        scaler=tcr_scaler,     # ensure same scaling basis as train
        weighting="self_tuning",
        mutual=True,
        knn_cfg=cpm.KNNConfig(k=knn_k, use_ann_if_large=True, ann_threshold=5000),
    )

    # Build hetero data
    data = HeteroData()
    data["tcr"].x = torch.tensor(tcr_feats_all, dtype=torch.float32)
    data["epitope"].x = torch.tensor(epi_feats_all, dtype=torch.float32)

    # similar edges
    if tcr_edge_index is None or tcr_edge_index.size == 0:
        data["tcr", "similar", "tcr"].edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        data["tcr", "similar", "tcr"].edge_index = torch.tensor(tcr_edge_index, dtype=torch.long)
        # weights are optional; the model in training didn't consume them explicitly

    # binds edges: TRAIN ONLY
    if len(train_df) > 0:
        tcr_epi = torch.tensor(
            train_df[["tcr_index", "epitope_index"]].values.T,
            dtype=torch.long,
        )
        epi_tcr = torch.tensor(
            train_df[["epitope_index", "tcr_index"]].values.T,
            dtype=torch.long,
        )
    else:
        tcr_epi = torch.empty((2, 0), dtype=torch.long)
        epi_tcr = torch.empty((2, 0), dtype=torch.long)

    data["tcr", "binds", "epitope"].edge_index = tcr_epi
    data["epitope", "rev_binds", "tcr"].edge_index = epi_tcr

    data = data.to(device)

    x_dict = {
        "tcr": data["tcr"].x,
        "epitope": data["epitope"].x,
    }
    edge_index_dict = {
        ("tcr", "similar", "tcr"): data["tcr", "similar", "tcr"].edge_index,
        ("tcr", "binds", "epitope"): data["tcr", "binds", "epitope"].edge_index,
        ("epitope", "rev_binds", "tcr"): data["epitope", "rev_binds", "tcr"].edge_index,
    }

    # Build model and load weights
    model = HeteroGNN(
        tcr_in=int(ckpt["tcr_in"]),
        epi_in=int(ckpt["epi_in"]),
        hidden=hidden,
        num_layers=num_layers,
    ).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    with torch.no_grad():
        out_dict = model(x_dict, edge_index_dict)

    return {
        "device": device,
        "model": model,
        "out_dict": out_dict,
        "tcr_to_idx": tcr_to_idx,
        "epi_to_idx": epi_to_idx,
        "pred_prepped": pred_prepped,
    }

def score_pairs_from_embeddings(dfp: pd.DataFrame, bundle: dict) -> pd.DataFrame:
    """Compute pred_prob for each (cdr3, peptide) row using embeddings + head."""
    device = bundle["device"]
    model = bundle["model"]
    out_dict = bundle["out_dict"]
    tcr_to_idx = bundle["tcr_to_idx"]
    epi_to_idx = bundle["epi_to_idx"]

    if len(dfp) == 0:
        out = dfp.copy()
        out["pred_prob"] = np.array([], dtype=float)
        return out

    dfp = dfp.copy()
    dfp["tcr_index"] = dfp["cdr3_truncated"].map(tcr_to_idx).astype(int)
    dfp["epitope_index"] = dfp["peptide"].map(epi_to_idx).astype(int)

    t_idx = torch.tensor(dfp["tcr_index"].values, dtype=torch.long, device=device)
    e_idx = torch.tensor(dfp["epitope_index"].values, dtype=torch.long, device=device)

    with torch.no_grad():
        t_emb = out_dict["tcr"][t_idx]
        e_emb = out_dict["epitope"][e_idx]
        pair_emb = torch.cat([t_emb, e_emb], dim=1)
        logits = model.head(pair_emb).squeeze(-1)
        probs = torch.sigmoid(logits).detach().cpu().numpy()

    dfp["pred_prob"] = probs
    return dfp

def attach_predictions_to_original(original_df: pd.DataFrame, predicted_prepped: pd.DataFrame) -> pd.DataFrame:
    """
    Merge predictions back to the original df (same shape as input),
    based on (cdr3, peptide). Rows that failed preprocessing get NaN pred_prob.
    """
    out = original_df.copy()
    out = out.merge(
        predicted_prepped[["cdr3", "peptide", "pred_prob"]],
        on=["cdr3", "peptide"],
        how="left",
    )
    return out

# -------------------------
# Your dataset creation
# -------------------------
tsv1 = pd.read_csv("train.tsv", sep="\t")
tsv2 = pd.read_csv("test.tsv", sep="\t")
csv_df = pd.read_csv("../files/combined_data.csv")
tsv_all = pd.concat([tsv1, tsv2], ignore_index=True)

tsv_cdr3_set = set(tsv_all["cdr3"])
tsv_peptide_set = set(tsv_all["peptide"])

cdr3_only_csv = csv_df[~csv_df["cdr3"].isin(tsv_cdr3_set)]
peptide_only_csv = csv_df[~csv_df["peptide"].isin(tsv_peptide_set)]
cdr3_and_peptide_only_csv = csv_df[
    (~csv_df["cdr3"].isin(tsv_cdr3_set)) &
    (~csv_df["peptide"].isin(tsv_peptide_set))
]

# -------------------------
# Inductive GNN prediction for each dataset
# -------------------------
predict_dfs = {
    "cdr3_only_csv": cdr3_only_csv,
    "peptide_only_csv": peptide_only_csv,
    "cdr3_and_peptide_only_csv": cdr3_and_peptide_only_csv,
    "csv_all": csv_df.sample(n=10000, random_state=0)
}

bundle = build_inductive_graph_and_embeddings(
    train_path="train.tsv",                 # train graph source
    predict_dfs=predict_dfs,
    model_path="models/tcr_epitope_model_inductive.pt",
    hidden=256,
    num_layers=2,
)

def predict_fn_with_bundle(df_pairs: pd.DataFrame) -> pd.DataFrame:
    dfp = prepare_pairs_df(df_pairs)                 # same preproc
    scored = score_pairs_from_embeddings(dfp, bundle) # adds pred_prob
    return attach_predictions_to_original(df_pairs, scored)  # merges back

datasets = {
    "cdr3_only_csv": cdr3_only_csv,
    "peptide_only_csv": peptide_only_csv,
    "cdr3_and_peptide_only_csv": cdr3_and_peptide_only_csv,
    "csv_all": csv_df.sample(n=10000, random_state=0),  
}


# Score each dataset
cdr3_only_prepped = score_pairs_from_embeddings(bundle["pred_prepped"]["cdr3_only_csv"], bundle)
peptide_only_prepped = score_pairs_from_embeddings(bundle["pred_prepped"]["peptide_only_csv"], bundle)
both_only_prepped = score_pairs_from_embeddings(bundle["pred_prepped"]["cdr3_and_peptide_only_csv"], bundle)
all_prepped = score_pairs_from_embeddings(bundle["pred_prepped"]["csv_all"], bundle)

# Merge back to original shapes
cdr3_only_csv_pred = attach_predictions_to_original(cdr3_only_csv, cdr3_only_prepped)
peptide_only_csv_pred = attach_predictions_to_original(peptide_only_csv, peptide_only_prepped)
cdr3_and_peptide_only_csv_pred = attach_predictions_to_original(cdr3_and_peptide_only_csv, both_only_prepped)
all_csv_pred = attach_predictions_to_original(datasets["csv_all"], all_prepped)

print(cdr3_only_csv_pred[["cdr3", "peptide", "pred_prob"]].head())
print(peptide_only_csv_pred[["cdr3", "peptide", "pred_prob"]].head())
print(cdr3_and_peptide_only_csv_pred[["cdr3", "peptide", "pred_prob"]].head())
print(all_csv_pred[["cdr3", "peptide", "pred_prob"]].head())

/tmp/ipykernel_3394198/2942631640.py:91: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(model_path, map_location=device)


             cdr3    peptide  pred_prob
0  CASSEGWHSYEQYF  KLGGALQAK   0.008916
1  CASSEGWHSYEQYF  KLGGALQAK   0.008916
2  CASSEGWHSYEQYF  KLGGALQAK   0.008916
3  CASGLNIDGDEQFF  KLGGALQAK   0.006779
4  CASGLNIDGDEQFF  KLGGALQAK   0.006779
               cdr3     peptide  pred_prob
0   CASSIVQGSNQPQHF  SLFNTVATLY   0.602911
1   CASSIVQGSNQPQHF  SLFNTVATLY   0.602911
2    CATSTGDSNQPQHF  SLFNTVATLY   0.990441
3    CATSTGDSNQPQHF  SLFNTVATLY   0.990441
4  CASSFPPQLGAYEQYF  SLFNTVATLY   0.999999
               cdr3     peptide  pred_prob
0   CASSIVQGSNQPQHF  SLFNTVATLY   0.602911
1   CASSIVQGSNQPQHF  SLFNTVATLY   0.602911
2  CASSFPPQLGAYEQYF  SLFNTVATLY   0.999999
3  CASSFPPQLGAYEQYF  SLFNTVATLY   0.999999
4  CASGLGTVASTDTQYF  SLFNTVATLY   0.907781
             cdr3        peptide  pred_prob
0  CASSLGRVGEKLFF      WICLLQFAY   0.999999
1   CASSSQGTGGYTF  APHGVVFLHVTYV   0.131418
2   CASSIRSSYEQYF      GILGFVFTL   1.000000
3   CASSIRSSYEQYF      GILGFVFTL   1.000000
4   CASSIRSSYEQYF      G

In [4]:
import os
import pandas as pd

def save_preds_and_rankings(
    df_pred: pd.DataFrame,
    name: str,
    out_dir: str = "pred_outputs",
    threshold: float = 0.5,
    top_k: int = 20,
):
    """
    Saves:
      1) full predictions CSV: <name>_preds.csv
      2) top-k per peptide CSV: <name>_topk_per_peptide.csv
      3) top-k per cdr3 CSV: <name>_topk_per_cdr3.csv
      4) high-confidence only (pred_prob >= threshold): <name>_highconf.csv
    """
    os.makedirs(out_dir, exist_ok=True)

    df = df_pred.copy()
    # ensure column exists
    if "pred_prob" not in df.columns:
        raise ValueError(f"{name}: expected column 'pred_prob' in df_pred")

    # label + clean
    df["pred_label"] = (df["pred_prob"] >= threshold).astype(int)

    # 1) full preds
    full_path = os.path.join(out_dir, f"{name}_preds.csv")
    df.to_csv(full_path, index=False)

    # 2) high confidence
    high = df[df["pred_prob"].notna() & (df["pred_prob"] >= threshold)].copy()
    high_path = os.path.join(out_dir, f"{name}_highconf.csv")
    high.to_csv(high_path, index=False)

    # 3) top-k per peptide
    # keep only rows with a probability
    valid = df[df["pred_prob"].notna()].copy()

    topk_pep = (
        valid.sort_values(["peptide", "pred_prob"], ascending=[True, False])
             .groupby("peptide", as_index=False)
             .head(top_k)
    )
    topk_pep_path = os.path.join(out_dir, f"{name}_topk_per_peptide.csv")
    topk_pep.to_csv(topk_pep_path, index=False)

    # 4) top-k per cdr3
    topk_tcr = (
        valid.sort_values(["cdr3", "pred_prob"], ascending=[True, False])
             .groupby("cdr3", as_index=False)
             .head(top_k)
    )
    topk_tcr_path = os.path.join(out_dir, f"{name}_topk_per_cdr3.csv")
    topk_tcr.to_csv(topk_tcr_path, index=False)

    print(f"\n[{name}] saved:")
    print(" -", full_path)
    print(" -", high_path)
    print(" -", topk_pep_path)
    print(" -", topk_tcr_path)

    # return dataframes too if you want to use them immediately
    return df, high, topk_pep, topk_tcr


# ---- Run for your three datasets ----
cdr3_only_csv_pred_labeled, cdr3_only_high, cdr3_only_topk_pep, cdr3_only_topk_tcr = save_preds_and_rankings(
    cdr3_only_csv_pred,
    name="cdr3_only_csv",
    out_dir="pred_outputs",
    threshold=0.5,
    top_k=20,
)

peptide_only_csv_pred_labeled, peptide_only_high, peptide_only_topk_pep, peptide_only_topk_tcr = save_preds_and_rankings(
    peptide_only_csv_pred,
    name="peptide_only_csv",
    out_dir="pred_outputs",
    threshold=0.5,
    top_k=20,
)

both_only_csv_pred_labeled, both_only_high, both_only_topk_pep, both_only_topk_tcr = save_preds_and_rankings(
    cdr3_and_peptide_only_csv_pred,
    name="cdr3_and_peptide_only_csv",
    out_dir="pred_outputs",
    threshold=0.5,
    top_k=20,
)

all_csv_pred_labeled, all_high, all_topk_pep, all_topk_tcr = save_preds_and_rankings(
    all_csv_pred,
    name="csv_all",
    out_dir="pred_outputs",
    threshold=0.5,
    top_k=20,
)


[cdr3_only_csv] saved:
 - pred_outputs/cdr3_only_csv_preds.csv
 - pred_outputs/cdr3_only_csv_highconf.csv
 - pred_outputs/cdr3_only_csv_topk_per_peptide.csv
 - pred_outputs/cdr3_only_csv_topk_per_cdr3.csv

[peptide_only_csv] saved:
 - pred_outputs/peptide_only_csv_preds.csv
 - pred_outputs/peptide_only_csv_highconf.csv
 - pred_outputs/peptide_only_csv_topk_per_peptide.csv
 - pred_outputs/peptide_only_csv_topk_per_cdr3.csv

[cdr3_and_peptide_only_csv] saved:
 - pred_outputs/cdr3_and_peptide_only_csv_preds.csv
 - pred_outputs/cdr3_and_peptide_only_csv_highconf.csv
 - pred_outputs/cdr3_and_peptide_only_csv_topk_per_peptide.csv
 - pred_outputs/cdr3_and_peptide_only_csv_topk_per_cdr3.csv

[csv_all] saved:
 - pred_outputs/csv_all_preds.csv
 - pred_outputs/csv_all_highconf.csv
 - pred_outputs/csv_all_topk_per_peptide.csv
 - pred_outputs/csv_all_topk_per_cdr3.csv


In [5]:
# assume you already created 'bundle' with build_inductive_graph_and_embeddings(...)

def predict_fn_with_bundle(df_pairs: pd.DataFrame) -> pd.DataFrame:
    dfp = prepare_pairs_df(df_pairs)                 # same preproc
    scored = score_pairs_from_embeddings(dfp, bundle) # adds pred_prob
    return attach_predictions_to_original(df_pairs, scored)  # merges back

datasets = {
    "cdr3_only_csv": cdr3_only_csv,
    "peptide_only_csv": peptide_only_csv,
    "cdr3_and_peptide_only_csv": cdr3_and_peptide_only_csv,
    "csv_all": csv_df.sample(n=10000, random_state=0),
}

results = []
for name, d in datasets.items():
    res = evaluate_positive_only_dataset(
        positives_df=d,
        predict_fn=predict_fn_with_bundle,
        name=name,
        neg_ratio=1,          # 1:1 negatives
        threshold=0.5,
        seed=0,
    )
    results.append(res)

metrics_df = pd.DataFrame(results)
print(metrics_df)


        n     pos     neg  accuracy  precision    recall        f1   roc_auc  \
0  275803  137921  137882  0.456489   0.466593  0.606644  0.527481  0.445204   
1   86995   43505   43490  0.489499   0.492803  0.712976  0.582788  0.491041   
2   83517   41766   41751  0.488859   0.492481  0.723771  0.586134  0.491455   
3   18903    9458    9445  0.544887   0.537132  0.653838  0.589767  0.563425   

                     dataset  neg_ratio  
0              cdr3_only_csv          1  
1           peptide_only_csv          1  
2  cdr3_and_peptide_only_csv          1  
3                    csv_all          1  
